# Explore Robot Segmentation (Steps 1–3 + Meta SAM 3.1)

Standalone exploration of the AL5D `random-both-cameras-video` demopack:
1. **Step 1**: Episode inventory, moving vs. paused frame timelines, and frozen 6/2/2 split.
2. **Step 2**: Per-camera median background, MAD variation, and static positional control.
3. **Step 3**: Offline partial-label teacher (`motion`) compared directly against **Meta SAM 3.1** (`sam3.1_multiplex.pt`), including prompt sensitivity (`"robot arm"` vs. `"robot arm with gripper"` vs. `"robotic arm"`) and resolution (`644` vs. `1008`) checks on moving and paused frames.

In [ ]:
pack_path = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Demopacks/random-both-cameras-video"
result_dir = "/Users/lboloni/Documents/Develop/Data/BerryPicker-Flows/RobotSegmentation_pilot"
camera = "dev2"
recompute = False
pilot_episodes = None   # None -> all 6 train + 2 val; or list of episode names
max_frames = 150        # None -> full episode; set e.g. 150 for pilot
run_sam3 = True
sam3_checkpoint = "/Users/lboloni/Documents/Develop/Data/BerryPicker-data-external/sam3/sam3.1_multiplex.pt"
sam3_prompt = "robot arm"
sam3_conf = 0.25
sam3_imgsz = 644
sam3_step = 4           # sample every k-th frame for offline SAM 3.1 labeling (1 = all frames)
recompute_sam3 = False
device = "auto"
seed = 0

In [ ]:
import json
import sys
from pathlib import Path

# Locate BerryPicker/src without importing ExpRunFlow or BerryPicker core
src_root = Path.cwd().resolve()
while src_root.name != "src" and src_root.parent != src_root:
    if (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
        break
    src_root = src_root.parent
if not (src_root / "_experimental" / "_2026_11_robot_segmentation").is_dir():
    src_root = Path("/Users/lboloni/Documents/Develop/Github/BerryPicker/BerryPicker/src")
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

import cv2
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

from _experimental._2026_11_robot_segmentation.baselines import MedianBackgroundModel, StaticMaskControl
from _experimental._2026_11_robot_segmentation.data import (
    BG, FG, UNKNOWN,
    camera_video_path, detect_pause_flags, frame_motion_scores,
    inventory_demopack, load_label_png, load_split, make_episode_split,
    read_video_frames, save_split,
)
from _experimental._2026_11_robot_segmentation.evaluate import extract_annotation_pack
from _experimental._2026_11_robot_segmentation.teachers import (
    MotionLabelTeacher, SAM3DirectSegmenter, SAM3Teacher, save_teacher_outputs,
)
from _experimental._2026_11_robot_segmentation.viz import make_comparison_strip, plot_episode_inventory

out = Path(result_dir)
out.mkdir(parents=True, exist_ok=True)

def show_rgb(rgb):
    bgr = cv2.cvtColor(np.asarray(rgb, dtype=np.uint8), cv2.COLOR_RGB2BGR)
    display(Image(data=cv2.imencode(".png", bgr)[1].tobytes()))

print("pack_path      :", pack_path)
print("result_dir     :", out)
print("camera         :", camera)
print("sam3_checkpoint:", sam3_checkpoint, "(exists=" + str(Path(sam3_checkpoint).is_file()) + ")")

## Step 1 — Episode Inventory & Frozen 6/2/2 Split

In [ ]:
inv_path = out / "inventory.json"
split_path = out / "split.json"
if recompute or not inv_path.is_file():
    inv = inventory_demopack(pack_path, cameras=("dev2", "dev3"), eval_frames_per_episode=12, seed=seed)
    inv_path.write_text(json.dumps(inv, indent=2) + "\n", encoding="utf-8")
    split = make_episode_split(pack_path, camera="dev2")
    save_split(split, split_path)
    extract_annotation_pack(inv, out / "eval_pack", camera=camera)
else:
    inv = json.loads(inv_path.read_text(encoding="utf-8"))
    split = load_split(split_path)

print("Train (6):", split.train)
print("Val   (2):", split.val)
print("Test  (2):", split.test, "(kept sealed until Step 7)")

fig = plot_episode_inventory(inv, camera=camera)
fig.savefig(out / f"inventory_{camera}.png", dpi=130)
plt.show()

## Step 2 — Median-Background Baseline & Static Positional Control

In [ ]:
bg_dir = out / "median_background" / camera
static_dir = out / "static_control" / camera

if recompute or not (bg_dir / "config.json").is_file():
    bg = MedianBackgroundModel.fit_from_demopack(pack_path, split.train, camera=camera, frame_step=10, threshold=0.12, dilate=1)
    bg.save(bg_dir)
    train_masks = []
    for ep in split.train:
        _, frms = read_video_frames(camera_video_path(pack_path, ep, camera), step=10)
        train_masks.extend(bg.segment(f) for f in frms)
    static = StaticMaskControl.fit_from_masks(train_masks, threshold=0.5, camera_id=camera)
    static.save(static_dir)
else:
    bg = MedianBackgroundModel.load(bg_dir)
    static = StaticMaskControl.load(static_dir)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))
axes[0].imshow(bg.background_rgb)
axes[0].set_title(f"Median background ({camera})")
axes[1].imshow(bg.mad_rgb.max(axis=2), cmap="inferno")
axes[1].set_title("Per-pixel MAD (variation)")
axes[2].imshow(static.occupancy, cmap="viridis", vmin=0, vmax=1)
axes[2].set_title("Static occupancy (train)")
for ax in axes:
    ax.axis("off")
fig.tight_layout()
fig.savefig(out / f"baselines_{camera}.png", dpi=130)
plt.show()

## Step 3 — Offline Teachers: Motion Partial Labels vs. Meta SAM 3.1

Green = confident foreground (`FG=1`), Amber = ambiguous/unknown (`UNKNOWN=255`, excluded from training loss), uncolored = confident background (`BG=0`).

In [ ]:
episodes_to_label = list(pilot_episodes) if pilot_episodes is not None else (split.train + split.val)
motion_teacher = MotionLabelTeacher(bg)
motion_summaries = {}
motion_results = {}

for ep in episodes_to_label:
    manifest_path = out / "teachers" / "motion" / camera / ep / "manifest.json"
    if recompute or not manifest_path.is_file():
        ids, frms = read_video_frames(camera_video_path(pack_path, ep, camera), max_frames=max_frames)
        res = motion_teacher.label_episode(frms, ids, episode=ep, camera=camera)
        s = save_teacher_outputs(res, frms, out / "teachers", save_overlays_every=25)
        motion_results[ep] = res
    else:
        s = json.loads(manifest_path.read_text(encoding="utf-8"))
    motion_summaries[ep] = s

sam3_seg = None
sam3_summaries = {}
if run_sam3 and Path(sam3_checkpoint).is_file():
    sam3_seg = SAM3DirectSegmenter(
        checkpoint=sam3_checkpoint, prompt=sam3_prompt, conf=sam3_conf, imgsz=sam3_imgsz, device=device, camera_id=camera
    )
    sam3_seg.save(out / "models" / f"sam3_direct_{camera}")
    sam3_teacher = SAM3Teacher(
        prompt=sam3_prompt, checkpoint=sam3_checkpoint, conf=sam3_conf, imgsz=sam3_imgsz, device=device, segmenter=sam3_seg
    )
    for ep in episodes_to_label:
        manifest_path = out / "teachers" / "sam3" / camera / ep / "manifest.json"
        if recompute or recompute_sam3 or not manifest_path.is_file():
            ids, frms = read_video_frames(camera_video_path(pack_path, ep, camera), step=sam3_step, max_frames=(None if max_frames is None else max(1, max_frames // sam3_step)))
            mres = motion_results.get(ep)
            res = sam3_teacher.label_episode(frms, ids, mres, episode=ep, camera=camera)
            s = save_teacher_outputs(res, frms, out / "teachers", save_overlays_every=10)
        else:
            s = json.loads(manifest_path.read_text(encoding="utf-8"))
        sam3_summaries[ep] = s

rows = ["| Teacher | Episode | Split | Total | Retained | Moving retained | Pause retained |", "|---|---|---|---:|---:|---:|---:|"]
for tname, smap in [("motion", motion_summaries), ("sam3 (3.1)", sam3_summaries)]:
    for ep, s in smap.items():
        part = "train" if ep in split.train else "val"
        rows.append(
            f"| {tname} | {ep} | {part} | {s['total_frames']} | "
            f"{s['retained_frames']} ({100*s['retained_fraction']:.1f}%) | "
            f"{s['moving_retained']}/{s['moving_total']} | "
            f"{s['pause_retained']}/{s['pause_total']} |"
        )
display(Markdown("\n".join(rows)))

### Meta SAM 3.1 Prompt & Resolution Check (`644` vs. `1008`)

In [ ]:
val_ep = next(ep for ep in episodes_to_label if ep in split.val)
ids, frms = read_video_frames(camera_video_path(pack_path, val_ep, camera), max_frames=max_frames)
motion = frame_motion_scores(frms)
pauses, _ = detect_pause_flags(motion)

if sam3_seg is not None:
    probe_idx = len(frms) // 3
    probe_rgb = frms[probe_idx]
    prompt_masks = {}
    orig_prompt = sam3_seg.prompt
    for p in ["robot arm", "robot arm with gripper", "robotic arm"]:
        sam3_seg.prompt = p
        prompt_masks[f'SAM3.1 "{p}"'] = sam3_seg.segment(probe_rgb)
    sam3_seg.prompt = orig_prompt
    show_rgb(make_comparison_strip(probe_rgb, prompt_masks, badge=f"f{ids[probe_idx]} 644px"))

    # Compare native 1008px vs 644px on the same frame
    sam3_1008 = SAM3DirectSegmenter(
        checkpoint=sam3_checkpoint, prompt=sam3_prompt, conf=sam3_conf, imgsz=1008, device=device, camera_id=camera
    )
    m644, t644 = sam3_seg.segment_timed(probe_rgb)
    m1008, t1008 = sam3_1008.segment_timed(probe_rgb)
    show_rgb(make_comparison_strip(
        probe_rgb,
        {f"SAM3.1 644 ({t644['total_ms']:.0f}ms)": m644, f"SAM3.1 1008 ({t1008['total_ms']:.0f}ms)": m1008},
        badge=f"f{ids[probe_idx]}",
    ))
    del sam3_1008

### Side-by-Side Validation Frames: Moving vs. Paused (Including Meta SAM 3.1)

In [ ]:
move_idx = np.flatnonzero(~pauses)
pause_idx = np.flatnonzero(pauses)
picks = []
if len(move_idx):
    picks.extend(move_idx[np.linspace(0, len(move_idx) - 1, min(3, len(move_idx))).astype(int)].tolist())
if len(pause_idx):
    picks.extend(pause_idx[np.linspace(0, len(pause_idx) - 1, min(2, len(pause_idx))).astype(int)].tolist())

m_lbl_dir = out / "teachers" / "motion" / camera / val_ep / "training_labels"
for idx in picks:
    fidx = ids[idx]
    rgb = frms[idx]
    panels = {
        "Median-BG": bg.segment(rgb),
        "Static": static.segment(rgb),
        "Motion": load_label_png(m_lbl_dir / f"frame_{fidx:06d}.png"),
    }
    if sam3_seg is not None:
        panels["SAM 3.1"] = sam3_seg.segment(rgb)
    badge = f"f{fidx} {'PAUSE' if pauses[idx] else 'MOVE'}"
    show_rgb(make_comparison_strip(rgb, panels, badge=badge))